# The wide net: robotics around the sector (sample for Step 4)

**Innovation Management · Group 34 · Humanoid robots for Electrolux**

| | |
|---|---|
| **Goal** | Build the wide net of robotics families, with the sector of rule v3 inside it, as the sample on which Step 4 can learn which patents become valuable. |
| **Used by** | Steps 3-4 (Thomas and Leo): the file `outputs/wide_net/wide_net_families.csv`, described in Section 4. |
| **Based on** | Thomas's search (group branch `thomas240103-patch-1`, `src/thomas_retrieval.py`, copied without changes): 19 keywords and robot codes, all of B25J and B62D57 among them. |
| **Reads** | `../ids/sector_family_ids_v3_PROD.csv`, the list of the sector saved by Step 0 (Section 7.4). |
| **Writes** | `outputs/wide_net/` (stays on TIP: git ignores it). |
| **How to run** | Open the notebook from the `wide_net` folder, next to `ids/`. With `RUN_SEARCH = False` it reads the last complete run of the search and only asks PATSTAT for the years (a few minutes); `True` runs the whole search again (hours on PROD). |

**Why a wide net.** The value of patents is very skewed (Lesson 6): a few patents hold most of it. Among the 2,907 families of the sector, the valuable ones would be a few dozen, too few to learn from. In robotics they are thousands, and robotics patents are close enough to humanoid ones to teach something useful. A model of Step 4 can learn on the wide net and give its scores to the firms of the sector.

**Two changes to Thomas's run**, so that the net and the sector follow the same rules:

1. **Years.** The search runs to 2025, as the sector (Thomas's run stopped at 2022). The year of a family is its earliest priority year over its PI applications, as in Step 0; the window is 2000-2025, and 2023-2025 are incomplete.
2. **The sector inside the net.** The families of rule v3 that the search does not find (mostly titles in other languages without a robot code) are added. The column `in_sector_v3` marks the sector.

Thomas's classification of the net (platform, component, affine) is not needed here: inside the sector the subfields come from Step 0, Section 7.4.

In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
from sqlalchemy import func

ROOT = Path.cwd()                        # open this notebook from the wide_net folder
sys.path.insert(0, str(ROOT / "src"))
from prod_run import load_prod_run      # Thomas's: finds the last complete run
from wide_net import build_wide_net     # ours: years and the sector inside the net

ENV = "PROD"                             # Thomas's search keeps only complete PROD runs
RUN_SEARCH = False                       # True runs Thomas's search; False reuses the last complete run to 2025
PATSTAT_EDITION = ""                     # the PATSTAT edition shown in TIP, saved with the run
YEAR_MIN, YEAR_MAX = 2000, 2025          # the window of Step 0
LAST_COMPLETE_YEAR = 2022                # 2023-2025 are incomplete (publication lag)
DUMMY_ID_FROM = 900_000_000              # appln_id from this value on are artificial records (as in Step 0)
SECTOR_IDS_FILE = ROOT.parent / "ids" / f"sector_family_ids_v3_{ENV}.csv"
OUT = ROOT / "outputs" / "wide_net"
OUT.mkdir(parents=True, exist_ok=True)

## 1. Thomas's search, to 2025

The search terms and codes are in `config/thomas_prod_search.json`, unchanged. The last period ends in 2025: in Thomas's notebook a line cut it to 2022, and here it is gone. The run saves one folder per run in `outputs/thomas_retrieval/`, with the families found (`candidates.csv`), the route of each hit (`retrieval_hits.csv`) and the codes (`assigned_codes.csv`).

In [ ]:
settings = json.loads((ROOT / "config" / "thomas_prod_search.json").read_text(encoding="utf-8"))
assert settings["periods"][-1][1] == YEAR_MAX, "the search must reach the end of the window"
if RUN_SEARCH:
    from thomas_retrieval import search
    search(ROOT, ENV, settings, patstat_edition=PATSTAT_EDITION)

run, meta = load_prod_run(ROOT, end_year=YEAR_MAX)
net_ids = sorted(pd.read_csv(run / "candidates.csv", usecols=["docdb_family_id"])["docdb_family_id"].astype(int).unique())
routes = pd.read_csv(run / "retrieval_hits.csv", usecols=["docdb_family_id", "lane"]).drop_duplicates()
print(f"run: {run.name} | PATSTAT edition: {meta.get('patstat_edition')}")
print(f"families found by the search: {len(net_ids):,}")
print(routes.groupby("lane")["docdb_family_id"].nunique().rename("families by route").to_string())

## 2. The year of each family

The earliest priority year over the PI applications of the family, the same definition as Step 0. The family IDs travel to PATSTAT in chunks of 5,000.

In [ ]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN

patstat = PatstatClient(env=ENV)
db = patstat.orm()
sector_ids = sorted(
    pd.read_csv(SECTOR_IDS_FILE)["docdb_family_id"].astype(int))


def priority_years(ids, chunk=5000):
    "Earliest priority year of each family, over its PI applications."
    ids = [int(i) for i in ids]   # plain integers: BigQuery cannot send int64
    parts = []
    for start in range(0, len(ids), chunk):
        q = (db.query(
                TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
                func.min(TLS201_APPLN.earliest_filing_year)
                .label("priority_year"))
             .filter(
                TLS201_APPLN.docdb_family_id.in_(ids[start:start + chunk]),
                TLS201_APPLN.ipr_type == "PI",
                TLS201_APPLN.appln_id < DUMMY_ID_FROM)
             .group_by(TLS201_APPLN.docdb_family_id))
        parts.append(patstat.df(q))
    return pd.concat(parts, ignore_index=True)


t0 = time.time()
net_years = priority_years(net_ids)
sector_years = priority_years(sector_ids)
print(f"years: {len(net_years):,} families of the net, "
      f"{len(sector_years):,} of the sector ({time.time() - t0:.0f} s)")
assert len(sector_years) == len(sector_ids), \
    "a family of the sector has no PI application"

## 3. The sector inside the net

Every family of rule v3 must end up in the net with `in_sector_v3 = True`. The table by year shows the share of the net that is the sector; the incomplete years are marked and never read as a decline.

In [ ]:
wide, counts = build_wide_net(net_years, sector_years, YEAR_MIN, YEAR_MAX)
for name, value in counts.items():
    print(f"{name:52s} {value:>9,}")

checks = {
    "every family of the sector is in the net": set(sector_ids) <= set(wide["docdb_family_id"]),
    "one row per family": wide["docdb_family_id"].is_unique,
    f"every year is in {YEAR_MIN}-{YEAR_MAX}": wide["priority_year"].between(YEAR_MIN, YEAR_MAX).all(),
}
for name, ok in checks.items():
    print("OK    -" if ok else "CHECK -", name)

by_year = wide.groupby("priority_year").agg(families=("docdb_family_id", "size"), sector=("in_sector_v3", "sum"))
by_year["sector share (%)"] = (100 * by_year["sector"] / by_year["families"]).round(2)
by_year["incomplete"] = by_year.index > LAST_COMPLETE_YEAR
wide.to_csv(OUT / "wide_net_families.csv", index=False)
print(f"\nsaved {OUT / 'wide_net_families.csv'} (stays on TIP)")
by_year

**What the PROD run shows.** Thomas's search, rerun to 2025 (19 keywords and the robot codes, with all of B25J and B62D57, in CPC and IPC), finds **239,742 families**: 210,231 through IPC codes, 164,944 through CPC codes and 27,205 through keywords (a family can come through more than one route). Up to 2022 the net holds 176,308 families, almost exactly the 176,342 of Thomas's run to 2022: the rerun reproduces his search.

The search already finds 2,903 of the 2,907 families of rule v3: only 4 are added. 35 families with a priority year outside 2000–2025 are removed. The wide net has **239,711 families, 2,907 of them (1.2%) in the sector**.

Until 2022 the sector is a small and shrinking part of robotics: 1.0–1.4% of the net in 2000–2010, 0.4–0.7% in 2017–2022. Between 2013–2017 and 2018–2022 the net almost doubles (46,636 to 92,139 families, ×1.98), while the sector grows less (368 to 526, ×1.43): the same picture as Step 1 against robotics. In the incomplete years the share jumps to 0.8% (2023), 2.2% (2024) and 4.8% (2025): the humanoid wave is not just robotics growing.

For Step 4 this solves the problem of size. The families of 2010–2017, which have five complete years of citations, are 59,373 in the net against about 480 in the sector, so the top 10% of their year holds about 5,900 families instead of about 50.

**Through which route does the search reach the sector?** Thomas's notebook 02 (branch `codex/patent-landscape-final`) splits his sector by the route of the search: keywords, CPC codes, IPC codes, or more than one. We do the same for the 2,907 families of rule v3, by period.

In [ ]:
# Through which route does Thomas's search reach the families of the sector? (his notebook 02 does it for his sector)
lanes = routes.groupby("docdb_family_id")["lane"].agg(lambda s: " + ".join(sorted(set(s))))
sector_routes = pd.Series(sector_ids).map(lanes).fillna("not found (added from Step 0)")
sector_years_ = pd.Series(sector_ids).map(wide.set_index("docdb_family_id")["priority_year"])
period = pd.cut(sector_years_, [1999, 2009, 2016, 2022, 2025], labels=["2000-09", "2010-16", "2017-22", "2023-25"])
route_table = pd.crosstab(sector_routes.to_numpy(), period.to_numpy(), margins=True, margins_name="all")
route_table.index.name, route_table.columns.name = "route of the search", "priority years"
display(route_table)
only_keywords = (sector_routes == "keywords").mean()
print(f"sector families reached by keywords only: {only_keywords:.1%}; "
      f"by a code (CPC or IPC), alone or with keywords: {sector_routes.str.contains('CPC|IPC').mean():.1%}")

**What the PROD run shows.** Almost every family of the sector is reached through the keywords of the search: 2,869 of 2,907 (98.7%), alone or together with a code. The codes alone bring only 34 families (1.2%), 20 of them from 2000–2009. This is expected: rule v3 reads the title, and the keywords of the search and the words of the rule overlap.

The other direction matters more. **625 families (21.5%) are reached by keywords only**: they carry none of the robot codes of the search (all of B25J and B62D57 and the other CPC and IPC codes of `config/thomas_prod_search.json`). A search by robot codes alone would miss about one family of the sector in five, and more in recent years: 9.5% of those of 2000–2009, about 20% in 2010–2022, 24.4% in 2023–2025. So the keywords are what makes the sector complete, not the codes.

The CPC route reaches fewer families than the IPC route (57.6% against 76.7%). 588 families (20.2%) come through IPC and keywords but through no CPC code: 43.6% of those of 2000–2009, many of them without any CPC code (the reason why Steps 1 and 2 take the IPC symbols when CPC is missing), and 19.3% of those of 2023–2025, probably because CPC codes are given later than IPC codes. The four families not found by the search are the four that Step 0 adds.

## 4. What the net gives to Steps 3-4

`outputs/wide_net/wide_net_families.csv` has one row per family of the net (239,711 on PROD):

| Column | Meaning |
|---|---|
| `docdb_family_id` | the DOCDB family |
| `priority_year` | earliest priority year over its PI applications, 2000-2025, as in Step 0 |
| `in_sector_v3` | `True` for the 2,907 families of rule v3 (Step 0) |
| `found_by_search` | `False` for the 4 families of the sector that the search does not find and that we add |

The routes of the search (keywords, CPC, IPC) stay in `outputs/thomas_retrieval/<run>/retrieval_hits.csv`. The file holds no texts and no applicant names: what a model needs (citations, codes, offices...) is read from PATSTAT on TIP, family by family. Like every output, it stays on TIP.